***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [第 2 章：干涉测量的数学工具箱](2_0_introduction.ipynb)
    * 上一节：[2.12 补充专题：立体角与天球面积元素](2_12_solid_angle.ipynb)
    * 下一节：[2.14 补充专题：一维 CLEAN 的数学演示](2_14_CLEAN_in_1D.ipynb)

***


导入标准模块:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    from IPython.display import HTML
except ImportError:
    def HTML(*args, **kwargs):
        return None
%matplotlib inline
HTML('../style/course.css')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12


导入本节所需的专用模块:

In [ ]:
try:
    from IPython.display import HTML
except ImportError:
    def HTML(*args, **kwargs):
        return None
HTML('../style/code_toggle.html')


## 2.13 补充专题：球面三角学<a id='math:sec:st'></a> <!--\label{math:sec:st}-->

球面三角学更直接地服务于下一章的位置天文学与坐标变换。本节保留在这里，是为了让需要的读者可以提前熟悉天球几何的基本恒等式，例如大圆弧、球面三角形以及边角之间的约束关系。

从全书主线看，它应被视为过渡到第 3 章的几何准备，而不是傅里叶分析主线的一部分。因此，若你的当前目标是尽快进入可见度、成像和校准，也可以先把本节作为参考材料保留，等进入天球坐标章节后再回头细读。


大多数读者对平面三角学都已有直觉，但天文学真正面对的是定义在单位球面上的方向关系。在球面上，“直线”被大圆弧取代，而球面三角形则由三条大圆弧围成。若把三条边分别记为 $a,b,c$，其含义不再是普通长度，而是对应圆心角；三个顶角记为 $A,B,C$，表示这些大圆弧相交时形成的夹角。

一旦把几何对象换成单位球面，平面三角学中的许多熟悉公式就会被改写成球面版本。对天文学最重要的是下面几条基本恒等式：

<p class=conclusion>
  <font size=4> <b>球面三角恒等式</b></font>
  <br>
  <br>
</p>

球面余弦定理：$\cos b = \cos a \cos c + \sin a \sin c \cos B$  

球面正弦定理：$\sin b \sin A = \sin B \sin a$  

五部件公式：$\sin b \cos A = \cos a \sin c - \sin a\cos c\cos B$  

前两条与平面三角学中的正弦定理和余弦定理形式相似，但要始终记住：这里的边本身就是角量。上述公式中所用到的边和角，如 [图 2.13.1 &#10549;](#math:fig:spher_trig) 所示。

<div class=advice>
<b>说明：</b> 球面三角学在射电天文学中的最常见用途，是处理地平坐标、赤道坐标、时角与天顶角之间的转换。第 3 章进入位置天文学后，会更频繁地调用这里的公式。
</div>


<img src='figures/spher_trig.svg' width=40%>

**图 2.13.1**：球面三角形 $ABC$。<a id='pos:math:spher_trig'></a> <!--\label{math:fig:spher_trig}-->

对观测者而言，最常见的球面三角形通常由天极、天顶和目标天体三点构成。它把纬度、赤纬、时角、高度角、方位角等量组织在同一个几何框架里，因此很多“坐标变换公式”其实只是球面三角恒等式在特定三角形上的直接应用。

从教学顺序上说，把这一节放在第 2 章末尾，是为了让读者在进入第 3 章之前先有一个可查阅的几何工具表；从使用频率上说，它更像一本放在手边的参考页，而不是需要机械背诵的公式清单。


#### 代码演示：把球面三角直接变成观测几何

下面用“天极-天顶-天源”这一个最常见的球面三角形，直接计算高度角和方位角随时角变化的轨迹。这个例子并不是为了抢先讲第 3 章全部内容，而是帮助你看到：球面三角学并不会停留在纸面公式上，它马上就会变成望远镜指向和观测窗口的计算工具。


In [ ]:
lat_deg = 34.08
source_dec_deg = 25.0
hour_angle_h = np.linspace(-6, 6, 721)

lat = np.deg2rad(lat_deg)
dec = np.deg2rad(source_dec_deg)
H = np.deg2rad(15.0 * hour_angle_h)

cos_z = np.sin(lat) * np.sin(dec) + np.cos(lat) * np.cos(dec) * np.cos(H)
z = np.arccos(np.clip(cos_z, -1.0, 1.0))
elevation = np.rad2deg(np.pi / 2 - z)

sin_az = -np.cos(dec) * np.sin(H) / np.cos(np.pi / 2 - z)
cos_az = (np.sin(dec) - np.sin(lat) * np.cos(z)) / (np.cos(lat) * np.sin(z))
azimuth = (np.rad2deg(np.arctan2(sin_az, cos_az)) + 360.0) % 360.0
visible = elevation > 0.0

fig = plt.figure(figsize=(12, 5))
ax1 = fig.add_subplot(1, 2, 1)
ax1.plot(hour_angle_h, elevation, lw=2, color='tab:blue')
ax1.axhline(0, color='0.3', ls=':')
ax1.axvline(0, color='tab:red', ls='--', label='transit')
ax1.set_xlabel('Hour angle H [h]')
ax1.set_ylabel('Elevation [deg]')
ax1.set_title('Elevation from spherical trigonometry')
ax1.grid(alpha=0.3)
ax1.legend()

ax2 = fig.add_subplot(1, 2, 2, projection='polar')
ax2.set_theta_zero_location('N')
ax2.set_theta_direction(-1)
ax2.plot(np.deg2rad(azimuth[visible]), 90.0 - elevation[visible], lw=2, color='tab:orange')
ax2.set_rlim(90, 0)
ax2.set_title('Track in the local sky')

plt.tight_layout()
plt.show()


这正是第 3 章坐标变换的原型。只要球面三角的三条边与三个角定义清楚，时角、纬度、赤纬、高度角和方位角之间的关系都能系统地写出来。换句话说，第 3 章里那些看起来像“坐标公式表”的内容，背后真正的几何骨架就是这里。


***

* 下一节：[2.14 补充专题：一维 CLEAN 的数学演示](2_14_CLEAN_in_1D.ipynb)
